In [2]:
from pathlib import Path
import numpy as np
import cv2

import matplotlib as mpl
import matplotlib.pyplot as plt

from src.cilia_detection.utils import yolobbox2bbox
from tqdm.notebook import tqdm
import skimage

%matplotlib inline
mpl.rcParams['figure.dpi'] = 300

In [3]:
def cellprofiler_tiff_to_mask(img: np.ndarray):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, contour_mask = cv2.threshold(gray, 254, 255, cv2.THRESH_BINARY)
    binary_mask = skimage.morphology.remove_small_holes(contour_mask.astype(bool), area_threshold=10000, connectivity=2)
    label_img = skimage.measure.label(binary_mask, connectivity=2)

    return label_img

def save_cellprofiler_results(img_path: Path, mask_dir: Path, bbox_dir: Path):
    class_label = 0
    img = cv2.imread(img_path.as_posix())
    label_img = cellprofiler_tiff_to_mask(img)
    regions = skimage.measure.regionprops(label_img)

    cv2.imwrite((mask_dir / img_path.name).with_suffix(".png").as_posix(), label_img)
    with open((bbox_dir / img_path.name).with_suffix(".txt"), 'w') as f:
        for region in regions:
            ymin, xmin, ymax, xmax = region.bbox
            w, h = xmax - xmin, ymax - ymin
            annotation = f'{class_label} {xmin + w / 2} {ymin + h / 2} {w} {h}\n'
            f.write(annotation)

In [7]:
# generate bboxes txt files

class_label = 0
cell_profiler_results = Path("../../data/cilia_easy_difficult_dataset/difficult/results_cellprofiler")
tiff_dir = cell_profiler_results / "tiff"
npy_dir = cell_profiler_results / "npy"
bbox_dir = cell_profiler_results / "bboxes"
bbox_dir.mkdir(exist_ok=True, parents=True)
mask_dir = cell_profiler_results / "labels"
mask_dir.mkdir(exist_ok=True, parents=True)

for p in tqdm(tiff_dir.glob("*.tiff")):
    save_cellprofiler_results(p, mask_dir, bbox_dir)

0it [00:00, ?it/s]

In [8]:
# generate images with bboxes (Yolo-format)

img_bbox_dir = cell_profiler_results / "images_bboxes"
img_bbox_dir.mkdir(exist_ok=True)

for img_p in tqdm((cell_profiler_results.parent / "converted_png").glob("*.png")):
    img = cv2.imread(img_p.as_posix())
    with open((bbox_dir / img_p.name).with_suffix(".txt")) as file:
        for line in file.readlines():
            yolo_bbox = [float(x) for x in line.strip().split(' ')[1:]]
            bbox = [int(x) for x in yolobbox2bbox(yolo_bbox)]
            cv2.rectangle(img, (bbox[0], bbox[1]), (bbox[2], bbox[3]), (255, 255, 255), thickness=2)

    cv2.imwrite((img_bbox_dir / img_p.name).with_suffix(".png").as_posix(), img)

0it [00:00, ?it/s]

In [ ]:
# # generate images with feature

# img_feature_dir = cell_profiler_results / "images_perimeter"
# output_dir.mkdir(exist_ok=True)

# for p in image_path.glob("*_Perimeter.npy"):
#     img = np.load(p)
#     img = (img * 255).astype(np.uint8)
#     kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3,3))
#     erode = cv2.erode(img, kernel, iterations=1)
    
#     bgr_erode = cv2.cvtColor(erode, cv2.COLOR_BGR2RGB)
#     cv2.imwrite((output_dir / p.name).with_suffix(".png").as_posix(), bgr_erode)